In [ ]:
# !python -m spacy download en_core_web_lg

In [ ]:
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm
import spacy

nlp_lg = spacy.load('en_core_web_lg')

In [ ]:
df = pd.read_parquet("gc_human_tweets.parquet")
df

**Supported NE Types**

- **PERSON:** People, including fictional
- **NORP:** Nationalities or religious or political groups
- **FAC:** Buildings, airports, highways, bridges, etc.
- **ORG:** Companies, agencies, institutions, etc.
- **GPE:** Countries, cities, states
- **LOC:** Non-GPE locations, mountain ranges, bodies of water
- **PRODUCT:** Objects, vehicles, foods, etc. (not services)
- **EVENT:** Named hurricanes, battles, wars, sports events, etc.
- **WORK_OF_ART:** Titles of books, songs, etc.
- **LAW:** Named documents made into laws
- **LANGUAGE:** Any named language
- **DATE:** Absolute or relative dates or periods
- **TIME:** Times smaller than a day
- **PERCENT:** Percentage, including "%"
- **MONEY:** Monetary values, including unit
- **QUANTITY:** Measurements, as of weight or distance
- **ORDINAL:** "first", "second", etc.
- **CARDINAL:** Numerals that do not fall under another type

In [ ]:
# get NE
all_entities = {
    'PERSON': [],
    'ORG': [],
    'GPE': [],
    'NORP': [],
    'EVENT': [],
    'LAW': [],
}

batch_size = 1000
for i in tqdm(range(0, len(df), batch_size)):
    batch = df['preprocessed_content'].iloc[i:i+batch_size]
    
    for tweet in batch:
        doc = nlp_lg(tweet)
        
        for ent in doc.ents:
            if ent.label_ in all_entities:
                # Filter out MeToo artifacts from PERSON
                if ent.label_ == 'PERSON' and ent.text in ['MeToo', 'Metoo', '#MeToo']:
                    continue
                all_entities[ent.label_].append(ent.text)

# Count each type
entity_counts = {
    entity_type: Counter(entities) 
    for entity_type, entities in all_entities.items()
}

# Show top 30 for each type
for entity_type, counts in entity_counts.items():
    print(f"\n=== {entity_type} (top 30) ===")
    print(f"Total mentions: {len(all_entities[entity_type]):,}")
    print(f"Unique: {len(counts):,}")
    for entity, count in counts.most_common(30):
        print(f"  {entity}: {count}")